# glass6: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('22_glass6_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,80.551,78.515,100.000,65.333,100.000,82.667,93.195,97.910,79.533
smote,84.454,81.864,96.000,72.000,99.459,85.730,92.587,97.279,83.159
bsmote,82.634,80.348,97.500,68.667,99.730,84.198,92.334,97.018,81.491
ros,85.538,84.273,100.000,73.667,100.000,86.833,93.268,98.000,84.905
adasyn,84.456,81.697,95.000,72.000,99.459,85.730,91.965,97.072,83.076
casb,87.373,86.273,100.000,77.000,100.000,88.500,92.332,97.423,86.823
casb_relaxed,86.184,83.030,93.500,75.333,99.189,87.261,92.591,97.288,84.607
casb_edited,85.967,82.091,91.833,75.333,98.919,87.126,92.434,97.288,84.029
casb_ros,88.126,85.697,95.000,78.667,99.459,89.063,92.312,97.378,86.911


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,84.170,80.333,93.333,72.000,98.919,85.459,89.929,94.514,82.252
smote,88.698,84.591,90.833,80.333,98.649,89.491,85.403,93.833,86.645
bsmote,85.638,79.333,85.000,75.333,97.838,86.586,86.141,94.072,82.486
ros,89.557,84.848,88.333,82.000,98.378,90.189,85.864,94.027,87.203
adasyn,89.015,81.987,83.452,82.000,97.297,89.649,85.122,93.680,85.501
casb,88.561,83.182,86.667,80.333,98.108,89.221,85.864,94.027,85.871
casb_relaxed,87.722,82.848,88.333,78.667,98.378,88.523,84.570,93.698,85.285
casb_edited,87.991,80.373,81.524,80.333,97.027,88.680,80.237,93.446,84.182
casb_ros,88.561,83.182,86.667,80.333,98.108,89.221,86.141,94.072,85.871


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,86.563,82.303,90.667,76.333,98.649,87.491,95.018,98.572,84.433
smote,94.139,90.606,91.667,90.000,98.649,94.324,94.459,97.545,92.373
bsmote,90.594,87.364,93.333,83.333,98.919,91.126,94.462,97.860,88.979
ros,91.569,88.606,93.333,85.000,98.919,91.959,95.145,97.928,90.088
adasyn,93.625,87.774,86.429,90.000,97.568,93.784,93.689,97.275,90.700
casb,94.275,91.515,93.333,90.000,98.919,94.459,94.920,98.491,92.895
casb_relaxed,93.268,89.697,91.667,88.333,98.649,93.491,94.705,98.378,91.483
casb_edited,92.385,88.636,91.667,86.667,98.649,92.658,94.569,97.297,90.511
casb_ros,91.693,89.364,95.000,85.000,99.189,92.095,95.755,98.784,90.528


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,83.974,78.939,88.667,72.000,98.378,85.189,74.100,91.423,81.457
smote,85.147,77.142,79.786,75.333,97.027,86.180,74.226,91.198,81.145
bsmote,85.412,78.112,81.952,75.333,97.297,86.315,72.924,88.865,81.762
ros,85.349,78.514,83.548,75.333,97.568,86.450,73.762,91.865,81.932
adasyn,85.719,75.998,75.762,77.000,96.216,86.608,73.112,87.162,80.858
casb,84.875,75.604,76.929,75.333,96.486,85.910,73.425,89.000,80.239
casb_relaxed,84.982,76.270,78.429,75.333,96.757,86.045,73.235,89.072,80.626
casb_edited,87.256,76.475,73.643,80.333,95.405,87.869,72.518,92.694,81.865
casb_ros,85.825,76.664,77.262,77.000,96.486,86.743,73.315,87.838,81.245


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,83.814,80.023,93.167,71.417,98.986,85.202,88.061,95.605,81.919
smote,88.110,83.551,89.571,79.417,98.446,88.931,86.669,94.964,85.830
bsmote,86.069,81.289,89.446,75.667,98.446,87.056,86.466,94.454,83.679
ros,88.003,84.060,91.304,79.000,98.716,88.858,87.010,95.455,86.032
adasyn,88.204,81.864,85.161,80.250,97.635,88.943,85.972,93.797,85.034
casb,88.771,84.143,89.232,80.667,98.378,89.523,86.635,94.735,86.457
casb_relaxed,88.039,82.962,87.982,79.417,98.243,88.830,86.275,94.609,85.500
casb_edited,88.400,81.894,84.667,80.667,97.500,89.083,84.939,95.181,85.147
casb_ros,88.551,83.727,88.482,80.250,98.311,89.280,86.881,94.518,86.139


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_ros,88.126,ros,85.538,2.588,WIN
1,f1,casb,86.273,ros,84.273,2.000,WIN
2,precision,casb,100.000,ros,100.000,0.000,TIE
3,recall,casb_ros,78.667,ros,73.667,5.000,WIN


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb,88.561,ros,89.557,-0.995,LOSS
1,f1,casb,83.182,ros,84.848,-1.667,LOSS
2,precision,casb_relaxed,88.333,smote,90.833,-2.500,LOSS
3,recall,casb,80.333,ros,82.000,-1.667,LOSS


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb,94.275,smote,94.139,0.136,WIN
1,f1,casb,91.515,smote,90.606,0.909,WIN
2,precision,casb_ros,95.000,bsmote,93.333,1.667,WIN
3,recall,casb,90.000,smote,90.000,0.000,TIE


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb,88.771,adasyn,88.204,0.567,WIN
1,f1,casb,84.143,ros,84.060,0.083,WIN
2,precision,casb,89.232,ros,91.304,-2.071,LOSS
3,recall,casb,80.667,adasyn,80.250,0.417,WIN


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.816497,0.800000,1.000000,0.666667,1.000000,0.833333,0.958333,0.990991,0.808248,None
1,42,1,svm,smote,0.50,0.816497,0.800000,1.000000,0.666667,1.000000,0.833333,0.958333,0.990991,0.808248,None
2,42,1,svm,bsmote,0.50,0.816497,0.800000,1.000000,0.666667,1.000000,0.833333,0.958333,0.990991,0.808248,None
3,42,1,svm,ros,0.50,0.816497,0.800000,1.000000,0.666667,1.000000,0.833333,0.976190,0.995495,0.808248,None
4,42,1,svm,adasyn,0.50,0.816497,0.800000,1.000000,0.666667,1.000000,0.833333,0.976190,0.995495,0.808248,None
5,42,1,svm,casb,0.50,0.816497,0.800000,1.000000,0.666667,1.000000,0.833333,0.958333,0.990991,0.808248,None
6,42,1,svm,casb_relaxed,0.50,0.816497,0.800000,1.000000,0.666667,1.000000,0.833333,0.976190,0.995495,0.808248,None
7,42,1,svm,casb_edited,0.50,0.816497,0.800000,1.000000,0.666667,1.000000,0.833333,0.976190,0.995495,0.808248,None
8,42,1,svm,casb_ros,0.50,0.912871,0.909091,1.000000,0.833333,1.000000,0.916667,0.958333,0.990991,0.910981,None
9,42,1,knn,original,0.50,0.816497,0.800000,1.000000,0.666667,1.000000,0.833333,0.856589,0.912162,0.808248,None
